# Block 2 — Frozen clinical knowledge graph

Catalogue, profile expansion, tube rules, `assume()`, `validate_request`.

Live Block 3 does **not** call `assume()` — that is Block 4. This notebook uses `KnowledgeGraph.load()` only. No PHI.


## 0. Download src/med_doc (zipball)


In [ ]:
# BOOTSTRAP_V6 — always refresh zipball (stale /content/epq3 lacks new kwargs like output_mode)
import importlib
import inspect
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

CONTENT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = CONTENT / "epq3"
SRC = REPO / "src"
URL = "https://codeload.github.com/RwaRwa599/epq3/zip/refs/heads/block1"

zpath = CONTENT / "epq3-block1.zip"
print("Downloading", URL)
urllib.request.urlretrieve(URL, zpath)
extract = CONTENT / "_epq3_extract"
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(extract)
found = list(extract.glob("*/src/med_doc/__init__.py"))
if not found:
    raise RuntimeError(f"zip missing src/med_doc: {list(extract.iterdir())}")
unpacked = found[0].parents[2]
if REPO.exists():
    shutil.rmtree(REPO)
shutil.move(str(unpacked), str(REPO))
shutil.rmtree(extract, ignore_errors=True)
zpath.unlink(missing_ok=True)

src = str(SRC.resolve())
while src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
os.chdir(REPO)
for name in list(sys.modules):
    if name == "med_doc" or name.startswith("med_doc."):
        del sys.modules[name]
importlib.invalidate_caches()
import med_doc
from med_doc.pipeline import run_blocks_1_to_5

print("BOOTSTRAP_V6")
print("cwd:", os.getcwd())
print("med_doc:", med_doc.__file__)
print("params:", list(inspect.signature(run_blocks_1_to_5).parameters))
from med_doc.htr.marks import TICK_POLICY
print("tick_policy:", TICK_POLICY)
if "output_mode" not in inspect.signature(run_blocks_1_to_5).parameters:
    raise RuntimeError(
        "stale med_doc (no output_mode). Runtime → Disconnect and delete runtime, "
        "re-open Run_in_Colab.ipynb from GitHub branch block1, then Run all."
    )
if TICK_POLICY != "slash-v2":
    raise RuntimeError(
        f"stale med_doc tick_policy={TICK_POLICY!r}. Disconnect and delete runtime, "
        "re-open Run_in_Colab.ipynb from GitHub branch block1."
    )


In [ ]:
# Runtime deps via pip CLI (not %pip / not pip -e — those restart Colab mid-run).
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pydantic", "matplotlib", "Pillow", "numpy"]
)


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet


## 1. Load `kg/lab_request_v1_kg.json`


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from med_doc.kg import KnowledgeGraph
from med_doc.paths import DEFAULT_KG

kg = KnowledgeGraph.load()
print("kg file:", DEFAULT_KG)
print("version:", kg.version)
print("catalogue items:", len(kg.catalogue))
print("profiles:", sorted(kg.profile_bundles)[:12], "...")


## 2. Profiles, tubes, aliases


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

print("lipid bundle:", kg.expand_profile("profile_lipid"))
print("cbc tubes:", kg.calculate_expected_tubes(["cbc"]))
print("cbc+glucose:", kg.calculate_expected_tubes(["cbc", "glucose_fasting"]))
print("resolve SGPT:", kg.resolve_alias("SGPT"))
ranked = kg.assume("others", "triglyc", {"ticked_ids": ["profile_lipid"]}, top_k=3)
for c in ranked:
    print(f"  assume {c.canonical_id!r:20} tier={c.tier} score={c.score:.3f} {c.reason}")


## 3. `validate_request` (observed tubes, not a copy of expected)


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

ok = kg.validate_request(["cbc"], observed_tubes={"EDTA": 1})
short = kg.validate_request(["cbc"], observed_tubes={"EDTA": 0})
missing = kg.validate_request(["cbc"], observed_tubes={})
print("match   valid", ok.is_valid, "expected", ok.expected_tubes)
print("short   valid", short.is_valid, short.discrepancies)
print("missing valid", missing.is_valid, "expected", missing.expected_tubes, "(empty obs is not auto-filled)")
